# FinSight Customer Analytics: Data Quality and Cleaning

## Objective
Identify data-quality problems and document cleaning decisions
before answering the business questions.

## Approach
- Preserve the original dataset.
- Check duplicates, missing values and inconsistent entries.
- Explain each cleaning decision.
- Keep unresolved issues visible.
- Use one analytical record per customer.

In [81]:
import sys
import pandas as pd

print("Python:", sys.executable)
print("Pandas:", pd.__version__)

Python: c:\Users\Tharindu\Desktop\FinSight-Customer-Analytics\.venv\Scripts\python.exe
Pandas: 3.0.6


In [82]:
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

file_path = (
    project_folder
    / "data"
    / "20260512_FintechTrainee_CaseStudy_Dataset_FinSightLanka_F.xlsx"
)

print("Dataset found:", file_path.exists())

Dataset found: True


In [83]:
raw = pd.read_excel(file_path, sheet_name="Customer_Data")

df = raw.copy()

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Unique customers:", df["Customer_ID"].nunique())

Rows: 508
Columns: 44
Unique customers: 500


In [84]:
print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Extra rows with repeated customer IDs:",
    df.duplicated(subset="Customer_ID").sum()
)

Exact duplicate rows: 3
Extra rows with repeated customer IDs: 8


## Duplicate handling

The dataset contains three fully identical duplicate rows.
These copies will be removed because they repeat the same information.

Other repeated customer IDs contain different values.
These records will be reviewed before choosing how to combine them.

In [85]:
df = raw.drop_duplicates().copy()

print("Rows after removing exact duplicates:", len(df))
print("Unique customers:", df["Customer_ID"].nunique())
print("Remaining extra customer rows:", df["Customer_ID"].duplicated().sum())

Rows after removing exact duplicates: 505
Unique customers: 500
Remaining extra customer rows: 5


In [86]:
duplicate_rows = df[
    df.duplicated(subset="Customer_ID", keep=False)
].copy()

print("Customers requiring review:")
print(duplicate_rows["Customer_ID"].unique())

Customers requiring review:
<StringArray>
['FSL-0089', 'FSL-0131', 'FSL-0201', 'FSL-0351', 'FSL-0421']
Length: 5, dtype: str


In [87]:
conflict_summary = []

for customer_id, group in duplicate_rows.groupby("Customer_ID"):
    for column in df.columns:
        if group[column].nunique(dropna=False) > 1:
            conflict_summary.append({
                "Customer_ID": customer_id,
                "Conflicting_Field": column
            })

conflict_summary = pd.DataFrame(conflict_summary)

display(conflict_summary)

,Customer_ID,Conflicting_Field
0,FSL-0089,Date_of_Birth
1,FSL-0089,Age
2,FSL-0131,KYC_Status
3,FSL-0201,Last_Login_Date
4,FSL-0351,NIC_Number
5,FSL-0421,NIC_Number


### Review of conflicting duplicates

Five customer IDs have conflicting information across six fields.

Matching values can be retained, but conflicting values require
a documented decision. Row order does not establish which record
is newer or more accurate.

NIC conflicts will remain flagged for identity verification.

In [88]:
for _, conflict in conflict_summary.iterrows():
    customer_id = conflict["Customer_ID"]
    column = conflict["Conflicting_Field"]

    if column == "NIC_Number":
        continue

    values = duplicate_rows.loc[
        duplicate_rows["Customer_ID"] == customer_id,
        column
    ].tolist()

    print(customer_id, "|", column, "|", values)

FSL-0089 | Date_of_Birth | [Timestamp('1890-03-11 00:00:00'), Timestamp('1982-06-13 00:00:00')]
FSL-0089 | Age | [134, 41]
FSL-0131 | KYC_Status | [nan, 'Pending']
FSL-0201 | Last_Login_Date | [Timestamp('2025-08-20 00:00:00'), Timestamp('2023-10-10 00:00:00')]


In [89]:
cleaning_log = [
    {
        "Customer_ID": "Multiple",
        "Field": "Entire row",
        "Action": "Removed 3 exact duplicate rows",
        "Reason": "All values were identical"
    }
]

In [90]:
# Start from the original data without identical copies.
reviewed = raw.drop_duplicates().copy()

# Flags record unresolved problems.
reviewed["Age_DOB_Conflict"] = False
reviewed["Login_Date_Conflict"] = False
reviewed["NIC_Conflict"] = False

# Conflicting age and birth date.
mask = reviewed["Customer_ID"] == "FSL-0089"

reviewed.loc[mask, "Age"] = float("nan")
reviewed.loc[mask, "Date_of_Birth"] = pd.NaT
reviewed.loc[mask, "Age_DOB_Conflict"] = True

# Retain the available KYC status.
mask = reviewed["Customer_ID"] == "FSL-0131"

reviewed.loc[mask, "KYC_Status"] = "Pending"

# Conflicting login dates.
mask = reviewed["Customer_ID"] == "FSL-0201"

reviewed.loc[mask, "Last_Login_Date"] = pd.NaT
reviewed.loc[mask, "Login_Date_Conflict"] = True

# Conflicting NIC numbers.
mask = reviewed["Customer_ID"].isin(["FSL-0351", "FSL-0421"])

reviewed.loc[mask, "NIC_Number"] = pd.NA
reviewed.loc[mask, "NIC_Conflict"] = True

In [91]:
duplicate_decisions = [
    {
        "Customer_ID": "FSL-0089",
        "Field": "Age and Date_of_Birth",
        "Action": "Set missing and flagged",
        "Reason": "Conflicting values; correct pair is unverified"
    },
    {
        "Customer_ID": "FSL-0131",
        "Field": "KYC_Status",
        "Action": "Retained Pending",
        "Reason": "Used the populated value; assumed it is applicable"
    },
    {
        "Customer_ID": "FSL-0201",
        "Field": "Last_Login_Date",
        "Action": "Set missing and flagged",
        "Reason": "Conflicting dates; snapshot date is unconfirmed"
    },
    {
        "Customer_ID": "FSL-0351",
        "Field": "NIC_Number",
        "Action": "Set missing and flagged",
        "Reason": "Identity information requires verification"
    },
    {
        "Customer_ID": "FSL-0421",
        "Field": "NIC_Number",
        "Action": "Set missing and flagged",
        "Reason": "Identity information requires verification"
    }
]

cleaning_log = cleaning_log[:1] + duplicate_decisions

display(pd.DataFrame(cleaning_log))

,Customer_ID,Field,Action,Reason
0,Multiple,Entire row,Removed 3 exact duplicate rows,All values were identical
1,FSL-0089,Age and Date_of_Birth,Set missing and flagged,Conflicting values; correct pair is unverified
2,FSL-0131,KYC_Status,Retained Pending,Used the populated value; assumed it is applic...
3,FSL-0201,Last_Login_Date,Set missing and flagged,Conflicting dates; snapshot date is unconfirmed
4,FSL-0351,NIC_Number,Set missing and flagged,Identity information requires verification
5,FSL-0421,NIC_Number,Set missing and flagged,Identity information requires verification


In [92]:
df = reviewed.drop_duplicates().reset_index(drop=True)

print("Rows:", len(df))
print("Unique customers:", df["Customer_ID"].nunique())
print("Remaining duplicate IDs:", df["Customer_ID"].duplicated().sum())

assert df["Customer_ID"].is_unique, "Some conflicting duplicates remain."

Rows: 500
Unique customers: 500
Remaining duplicate IDs: 0


In [93]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

display(
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Count", ascending=False)
)

,Missing_Count,Missing_Percentage
FD_Amount,355,71.0
Loan_Amount,283,56.6
Loan_Type,282,56.4
Loan_Tenure_Months,282,56.4
Monthly_Repayment_Amount,282,56.4
Outstanding_Loan_Balance,281,56.2
Loan_Repayment_Status,280,56.0
Loan_Start_Date,280,56.0
Referral_Source,247,49.4
Last_Login_Date,169,33.8


## Missing values and product consistency

Missing values are assessed using related product flags.

For example:
- A missing loan amount may be expected for a non-borrower.
- A missing loan amount for a borrower needs investigation.
- Loan details recorded for a non-borrower may indicate a contradiction.

Missing financial values will not automatically be replaced with zero.

In [94]:
loan_fields = [
    "Loan_Type",
    "Loan_Amount",
    "Loan_Tenure_Months",
    "Monthly_Repayment_Amount",
    "Loan_Repayment_Status",
    "Loan_Start_Date",
    "Outstanding_Loan_Balance"
]

loan_checks = []

for field in loan_fields:
    loan_checks.append({
        "Field": field,
        "Loan Yes: missing": (
            (df["Has_Loan"] == "Yes") & df[field].isna()
        ).sum(),
        "Loan No: populated": (
            (df["Has_Loan"] == "No") & df[field].notna()
        ).sum(),
        "Loan unknown: populated": (
            df["Has_Loan"].isna() & df[field].notna()
        ).sum()
    })

display(pd.DataFrame(loan_checks))

,Field,Loan Yes: missing,Loan No: populated,Loan unknown: populated
0,Loan_Type,2,1,1
1,Loan_Amount,3,1,1
2,Loan_Tenure_Months,2,1,1
3,Monthly_Repayment_Amount,2,1,1
4,Loan_Repayment_Status,2,3,1
5,Loan_Start_Date,2,3,1
6,Outstanding_Loan_Balance,2,2,1


In [95]:
other_checks = {
    "FD Yes but amount missing": (
        (df["Has_Fixed_Deposit"] == "Yes")
        & df["FD_Amount"].isna()
    ).sum(),

    "FD No but positive amount": (
        (df["Has_Fixed_Deposit"] == "No")
        & (df["FD_Amount"] > 0)
    ).sum(),

    "App Yes but login date missing": (
        (df["Mobile_App_User"] == "Yes")
        & df["Last_Login_Date"].isna()
    ).sum(),

    "App No but login date populated": (
        (df["Mobile_App_User"] == "No")
        & df["Last_Login_Date"].notna()
    ).sum()
}

display(
    pd.DataFrame(
        other_checks.items(),
        columns=["Check", "Customer_Count"]
    )
)

,Check,Customer_Count
0,FD Yes but amount missing,0
1,FD No but positive amount,0
2,App Yes but login date missing,1
3,App No but login date populated,0


In [96]:
missing_app_login = df.loc[
    (df["Mobile_App_User"] == "Yes")
    & df["Last_Login_Date"].isna(),
    ["Customer_ID", "Mobile_App_User", "Login_Date_Conflict"]
]

display(missing_app_login)

,Customer_ID,Mobile_App_User,Login_Date_Conflict
200,FSL-0201,Yes,True


### Fixed-deposit and app consistency checks

No missing FD amounts were found among customers marked as holding
a fixed deposit. No positive FD amounts were found among customers
marked as not holding one.

These checks found no contradictions under the conditions tested.
They do not prove that every recorded FD amount is accurate.

One app user has a missing login date. This is checked against the
previously documented duplicate-date conflict. App membership and
recent app activity are treated as different measures.

In [97]:
display(pd.DataFrame(loan_checks))

,Field,Loan Yes: missing,Loan No: populated,Loan unknown: populated
0,Loan_Type,2,1,1
1,Loan_Amount,3,1,1
2,Loan_Tenure_Months,2,1,1
3,Monthly_Repayment_Amount,2,1,1
4,Loan_Repayment_Status,2,3,1
5,Loan_Start_Date,2,3,1
6,Outstanding_Loan_Balance,2,2,1


In [98]:
# At least one loan detail is missing.
missing_loan_detail = df[loan_fields].isna().any(axis=1)

# At least one loan detail is present.
has_loan_detail = df[loan_fields].notna().any(axis=1)

needs_loan_review = (
    ((df["Has_Loan"] == "Yes") & missing_loan_detail)
    | ((df["Has_Loan"] == "No") & has_loan_detail)
    | df["Has_Loan"].isna()
)

loan_review = df.loc[
    needs_loan_review,
    ["Customer_ID", "Has_Loan"] + loan_fields
].copy()

print("Unique customers requiring review:", len(loan_review))

with pd.option_context("display.max_columns", None):
    display(loan_review)

Unique customers requiring review: 13


,Customer_ID,Has_Loan,Loan_Type,Loan_Amount,Loan_Tenure_Months,Monthly_Repayment_Amount,Loan_Repayment_Status,Loan_Start_Date,Outstanding_Loan_Balance
40,FSL-0041,Yes,Business,NaN,12.0,126788.37,On-Time,2021-07-26,228525.23
48,FSL-0049,No,NaN,NaN,NaN,NaN,NaN,2010-01-01,NaN
70,FSL-0071,No,NaN,NaN,NaN,NaN,Defaulted,NaT,150000.00
85,FSL-0086,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
95,FSL-0096,Yes,NaN,NaN,NaN,NaN,NaN,NaT,NaN
145,FSL-0146,No,Housing,1792951.23,48.0,41088.47,On-Time,2023-11-27,1516298.23
165,FSL-0166,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
225,FSL-0226,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
355,FSL-0356,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
385,FSL-0386,Yes,NaN,NaN,NaN,NaN,NaN,NaT,NaN


In [99]:
# Preserve the original Has_Loan column.
df["Has_Loan_Clean"] = df["Has_Loan"].fillna("Unknown")

df["Loan_Status_Basis"] = "Recorded flag"
df.loc[
    df["Has_Loan"].isna(),
    "Loan_Status_Basis"
] = "Unknown: missing flag"

# Positive balances support an inferred Yes.
inferred_ids = ["FSL-0071", "FSL-0146", "FSL-0476"]

mask = df["Customer_ID"].isin(inferred_ids)

df.loc[mask, "Has_Loan_Clean"] = "Yes"
df.loc[mask, "Loan_Status_Basis"] = "Inferred from positive balance"

# No flag and Delayed status cannot be reconciled.
mask = df["Customer_ID"] == "FSL-0431"

df.loc[mask, "Has_Loan_Clean"] = "Unknown"
df.loc[mask, "Loan_Status_Basis"] = "Unresolved repayment-status conflict"

# Preserve the list of customers requiring source review.
df["Loan_Review_Flag"] = df["Customer_ID"].isin(
    loan_review["Customer_ID"]
)

In [100]:
loan_decisions = [
    ("FSL-0041", "Retained Yes; original amount remains missing",
     "Outstanding balance is available"),

    ("FSL-0071", "Inferred Yes",
     "Positive outstanding balance contradicts recorded No"),

    ("FSL-0146", "Inferred Yes",
     "Positive outstanding balance and supporting loan details"),

    ("FSL-0476", "Inferred Yes",
     "Missing flag but positive outstanding balance"),

    ("FSL-0096", "Retained recorded Yes",
     "Loan details missing; no evidence to reverse the flag"),

    ("FSL-0386", "Retained recorded Yes",
     "Loan details missing; no evidence to reverse the flag"),

    ("FSL-0049", "Retained recorded No; flagged for review",
     "Start date alone does not establish a current loan"),

    ("FSL-0391", "Retained recorded No; flagged for review",
     "Start date alone does not establish a current loan"),

    ("FSL-0431", "Set analytical status to Unknown",
     "No flag conflicts with Delayed status; balance missing")
]

for customer_id in ["FSL-0086", "FSL-0166", "FSL-0226", "FSL-0356"]:
    loan_decisions.append((
        customer_id,
        "Kept status Unknown",
        "Loan flag and all loan details are missing"
    ))

# Replace this section of the log if the cell is run again.
cleaning_log = [
    entry for entry in cleaning_log
    if entry.get("Stage") != "Loan review"
]

for customer_id, action, reason in loan_decisions:
    cleaning_log.append({
        "Stage": "Loan review",
        "Customer_ID": customer_id,
        "Field": "Loan information",
        "Action": action,
        "Reason": reason
    })

In [101]:
display(
    pd.crosstab(
        df["Has_Loan"].fillna("Unknown"),
        df["Has_Loan_Clean"],
        rownames=["Original status"],
        colnames=["Analytical status"]
    )
)

print(
    "Inferred borrowers:",
    (df["Loan_Status_Basis"] == "Inferred from positive balance").sum()
)

print(
    "Unknown loan status:",
    (df["Has_Loan_Clean"] == "Unknown").sum()
)

print("Total customers:", len(df))

Analytical status,No,Unknown,Yes
Original status,,,
No,274,1,2
Unknown,0,4,1
Yes,0,0,218


Inferred borrowers: 3
Unknown loan status: 5
Total customers: 500


### Loan-status decisions

Three customers were classified as borrowers based on positive
outstanding balances. These classifications are inferred, not verified.

Two customers with recorded Yes flags but no loan details remain
borrowers for penetration analysis. Their missing balances and
repayment statuses remain unavailable for the relevant risk calculations.

Five customers have unknown analytical loan status.

Original loan flags and loan details have been preserved.
Final loan metrics will be compared with results using the original
flags to assess the effect of these decisions.

## Category standardisation

Text categories are checked for spelling differences, extra spaces
and inconsistent capitalisation.

Clear equivalents are standardised in new columns.
Original values are preserved, and missing values remain missing.

In [102]:
category_columns = [
    "Gender",
    "District",
    "Province",
    "Urban_Rural",
    "Customer_Segment",
    "Acquisition_Channel",
    "KYC_Status"
]

for column in category_columns:
    print("\n" + column)
    print(df[column].value_counts(dropna=False).to_string())


Gender
Gender
Female    258
Male      236
M           1
male        1
FEMALE      1
F           1
MALE        1
f           1

District
District
Jaffna         58
Kurunegala     55
Ratnapura      54
Gampaha        52
Badulla        50
Colombo        49
Trincomalee    49
Kandy          46
Matara         44
Galle          34
NaN             5
Columbo         1
Kanady          1
Gale            1
Jafna           1

Province
Province
Western          102
Southern          82
Northern          58
North Western     56
Sabaragamuwa      56
Uva               50
Eastern           50
Central           46

Urban_Rural
Urban_Rural
Urban       189
Suburban    188
Rural       123

Customer_Segment
Customer_Segment
Regular    313
Starter     91
Premium     91
NaN          5

Acquisition_Channel
Acquisition_Channel
Digital     131
Referral    129
Branch      123
Agent       117

KYC_Status
KYC_Status
Verified    351
Expired      73
Pending      70
NaN           6


In [103]:
for column in category_columns:
    df[column + "_Clean"] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.title()
    )

# Standardise gender abbreviations.
df["Gender_Clean"] = df["Gender_Clean"].replace({
    "M": "Male",
    "F": "Female"
})

# Correct clear district spelling errors.
df["District_Clean"] = df["District_Clean"].replace({
    "Columbo": "Colombo",
    "Kanady": "Kandy",
    "Gale": "Galle",
    "Jafna": "Jaffna"
})

In [104]:
# Avoid duplicate log entries when rerunning this cell.
cleaning_log = [
    entry for entry in cleaning_log
    if entry.get("Stage") != "Category standardisation"
]

category_changes = []

for column in category_columns:
    clean_column = column + "_Clean"

    original = df[column].astype("string")
    cleaned = df[clean_column]

    changed = original.ne(cleaned).fillna(False)

    category_changes.append({
        "Field": column,
        "Values_Changed": int(changed.sum())
    })

    for index in df.index[changed]:
        cleaning_log.append({
            "Stage": "Category standardisation",
            "Customer_ID": df.at[index, "Customer_ID"],
            "Field": column,
            "Action": (
                f"{df.at[index, column]} -> "
                f"{df.at[index, clean_column]}"
            ),
            "Reason": "Standardised formatting or clear spelling error"
        })

display(pd.DataFrame(category_changes))

,Field,Values_Changed
0,Gender,6
1,District,4
2,Province,0
3,Urban_Rural,0
4,Customer_Segment,0
5,Acquisition_Channel,0
6,KYC_Status,0


In [105]:
district_to_province = {
    "Colombo": "Western",
    "Gampaha": "Western",
    "Kandy": "Central",
    "Galle": "Southern",
    "Matara": "Southern",
    "Jaffna": "Northern",
    "Kurunegala": "North Western",
    "Ratnapura": "Sabaragamuwa",
    "Badulla": "Uva",
    "Trincomalee": "Eastern"
}

expected_province = df["District_Clean"].map(district_to_province)

df["District_Province_Conflict"] = (
    expected_province.notna()
    & df["Province_Clean"].notna()
    & df["Province_Clean"].ne(expected_province)
).fillna(False)

print(
    "District-province conflicts:",
    df["District_Province_Conflict"].sum()
)

print(
    "Missing districts:",
    df["District_Clean"].isna().sum()
)

display(
    df.loc[
        df["District_Province_Conflict"],
        ["Customer_ID", "District_Clean", "Province_Clean"]
    ]
)

District-province conflicts: 4
Missing districts: 5


,Customer_ID,District_Clean,Province_Clean
25,FSL-0026,Colombo,Southern
90,FSL-0091,Kandy,Southern
160,FSL-0161,Galle,North Western
340,FSL-0341,Jaffna,Southern


In [106]:
# Preserve the standardised source province.
df["Province_Analysis"] = df["Province_Clean"].copy()
df["Province_Basis"] = "Recorded province"

# Derive province only for the four conflicting pairs.
mask = df["District_Province_Conflict"]

df.loc[mask, "Province_Analysis"] = (
    df.loc[mask, "District_Clean"].map(district_to_province)
)

df.loc[mask, "Province_Basis"] = (
    "Derived from district; unverified assumption"
)

display(
    df.loc[
        mask,
        [
            "Customer_ID",
            "District_Clean",
            "Province_Clean",
            "Province_Analysis"
        ]
    ]
)

,Customer_ID,District_Clean,Province_Clean,Province_Analysis
25,FSL-0026,Colombo,Southern,Western
90,FSL-0091,Kandy,Southern,Central
160,FSL-0161,Galle,North Western,Southern
340,FSL-0341,Jaffna,Southern,Northern


In [107]:
cleaning_log = [
    entry for entry in cleaning_log
    if entry.get("Stage") != "Geography review"
]

for index in df.index[mask]:
    cleaning_log.append({
        "Stage": "Geography review",
        "Customer_ID": df.at[index, "Customer_ID"],
        "Field": "Province_Analysis",
        "Action": (
            f"{df.at[index, 'Province_Clean']} -> "
            f"{df.at[index, 'Province_Analysis']}"
        ),
        "Reason": (
            "Assumed recorded district is correct; "
            "original province and conflict flag preserved"
        )
    })

### Geography assumption

Four district–province pairs disagree.

For the main analysis, province is derived from the recorded district
for these four customers. This assumes the district is correct and
requires verification.

Province-level findings will also be checked with these customers
excluded to assess whether the assumption changes the conclusions.

Five missing districts remain unknown.

In [108]:
financial_columns = [
    "Savings_Balance",
    "Monthly_Deposit_Avg",
    "Monthly_Withdrawal_Avg",
    "Net_Monthly_Flow",
    "Loan_Amount",
    "Outstanding_Loan_Balance",
    "FD_Amount"
]

financial_checks = []

for column in financial_columns:
    financial_checks.append({
        "Field": column,
        "Missing": df[column].isna().sum(),
        "Negative": (df[column] < 0).sum(),
        "Zero": (df[column] == 0).sum(),
        "Minimum": df[column].min(),
        "Median": df[column].median(),
        "Maximum": df[column].max()
    })

display(pd.DataFrame(financial_checks).round(2))

,Field,Missing,Negative,Zero,Minimum,Median,Maximum
0,Savings_Balance,6,4,0,-88000.00,442763.82,1.000000e+09
1,Monthly_Deposit_Avg,6,0,0,194.79,17858.54,3.471401e+05
2,Monthly_Withdrawal_Avg,0,0,0,100.99,9684.80,9.999999e+06
3,Net_Monthly_Flow,0,0,0,30.87,6044.87,1.962410e+05
4,Loan_Amount,283,0,0,54308.53,1421377.63,2.998363e+06
5,Outstanding_Loan_Balance,281,0,0,16258.84,669510.68,2.460698e+06
6,FD_Amount,355,0,0,113217.78,1101160.81,1.977886e+06


In [109]:
display(
    df.nlargest(5, "Savings_Balance")[
        ["Customer_ID", "Savings_Balance"]
    ]
)

display(
    df.nlargest(5, "Monthly_Withdrawal_Avg")[
        [
            "Customer_ID",
            "Monthly_Deposit_Avg",
            "Monthly_Withdrawal_Avg",
            "Net_Monthly_Flow"
        ]
    ]
)

,Customer_ID,Savings_Balance
55,FSL-0056,1.000000e+09
221,FSL-0222,4.916005e+06
27,FSL-0028,4.864806e+06
371,FSL-0372,4.819645e+06
238,FSL-0239,4.784741e+06


,Customer_ID,Monthly_Deposit_Avg,Monthly_Withdrawal_Avg,Net_Monthly_Flow
125,FSL-0126,9205.64,9999999.00,3473.72
295,FSL-0296,8134.24,8888888.00,1488.34
307,FSL-0308,338960.85,230688.56,108272.29
356,FSL-0357,290928.67,220413.45,70515.22
201,FSL-0202,308090.44,219046.23,89044.21


In [110]:
suspicious_savings = df.loc[
    df["Customer_ID"] == "FSL-0056",
    "Savings_Balance"
].iloc[0]

print(f"Recorded savings: LKR {suspicious_savings:,.2f}")

Recorded savings: LKR 999,999,999.00


In [111]:
df["Savings_Negative_Flag"] = df["Savings_Balance"] < 0

df["Savings_Outlier_Review"] = (
    df["Customer_ID"] == "FSL-0056"
)

df["Savings_Analysis"] = df["Savings_Balance"].copy()

exclude_savings = (
    df["Savings_Negative_Flag"]
    | df["Savings_Outlier_Review"]
)

df.loc[exclude_savings, "Savings_Analysis"] = float("nan")

print("Negative savings:", df["Savings_Negative_Flag"].sum())
print("Outlier under review:", df["Savings_Outlier_Review"].sum())
print("Usable savings values:", df["Savings_Analysis"].notna().sum())

Negative savings: 4
Outlier under review: 1
Usable savings values: 489


In [112]:
df["Flow_Calculated"] = (
    df["Monthly_Deposit_Avg"]
    - df["Monthly_Withdrawal_Avg"]
)

comparable = (
    df["Flow_Calculated"].notna()
    & df["Net_Monthly_Flow"].notna()
)

df["Flow_Mismatch_Flag"] = (
    comparable
    & (
        (df["Flow_Calculated"] - df["Net_Monthly_Flow"]).abs()
        > 0.01
    )
)

display(
    df.loc[
        df["Flow_Mismatch_Flag"],
        [
            "Customer_ID",
            "Monthly_Deposit_Avg",
            "Monthly_Withdrawal_Avg",
            "Net_Monthly_Flow",
            "Flow_Calculated"
        ]
    ]
)

,Customer_ID,Monthly_Deposit_Avg,Monthly_Withdrawal_Avg,Net_Monthly_Flow,Flow_Calculated
125,FSL-0126,9205.64,9999999.0,3473.72,-9990793.36
295,FSL-0296,8134.24,8888888.0,1488.34,-8880753.76


In [113]:
df["Flow_Analysis"] = df["Flow_Calculated"].copy()

invalid_flow = (
    df["Monthly_Deposit_Avg"].isna()
    | df["Monthly_Withdrawal_Avg"].isna()
    | (df["Monthly_Deposit_Avg"] < 0)
    | (df["Monthly_Withdrawal_Avg"] < 0)
    | df["Flow_Mismatch_Flag"]
)

df.loc[invalid_flow, "Flow_Analysis"] = float("nan")

print("Contradictory flows:", df["Flow_Mismatch_Flag"].sum())
print("Usable flows:", df["Flow_Analysis"].notna().sum())
print("Unavailable flows:", df["Flow_Analysis"].isna().sum())

Contradictory flows: 2
Usable flows: 492
Unavailable flows: 8


In [114]:
# Both comparisons exclude negative balances.
nonnegative_savings = df.loc[
    df["Savings_Balance"] >= 0,
    "Savings_Balance"
]

comparison = pd.DataFrame({
    "Including suspicious balance": [
        nonnegative_savings.count(),
        nonnegative_savings.sum(),
        nonnegative_savings.mean(),
        nonnegative_savings.median()
    ],
    "Excluding suspicious balance": [
        df["Savings_Analysis"].count(),
        df["Savings_Analysis"].sum(),
        df["Savings_Analysis"].mean(),
        df["Savings_Analysis"].median()
    ]
}, index=["Valid count", "Total savings", "Average savings", "Median savings"])

display(comparison.round(2))

,Including suspicious balance,Excluding suspicious balance
Valid count,4.900000e+02,4.890000e+02
Total savings,1.378349e+09,3.783492e+08
Average savings,2.812958e+06,7.737203e+05
Median savings,4.485819e+05,4.482971e+05


### Financial-value treatment

Original financial values are preserved.

Negative savings and one unusually large savings balance are excluded
from the main savings analysis pending verification. Customers remain
available for analyses that do not require valid savings.

Savings summaries are compared with and without the large balance.

Monthly flow is calculated from deposit and withdrawal averages.
Where inputs are missing, negative, or contradict recorded net flow
by more than LKR 0.01, analytical flow remains unknown.

Disputed withdrawal amounts are not reconstructed from net flow
because the authoritative field is unknown.

In [115]:
display(comparison.style.format("{:,.2f}"))

,Including suspicious balance,Excluding suspicious balance
Valid count,490.00,489.00
Total savings,"1,378,349,215.79","378,349,216.79"
Average savings,"2,812,957.58","773,720.28"
Median savings,"448,581.87","448,297.06"


In [116]:
# Remove previous entries from this stage before rebuilding them.
cleaning_log = [
    entry for entry in cleaning_log
    if entry.get("Stage") != "Financial review"
]

for index in df.index[df["Savings_Negative_Flag"]]:
    cleaning_log.append({
        "Stage": "Financial review",
        "Customer_ID": df.at[index, "Customer_ID"],
        "Field": "Savings_Analysis",
        "Action": "Excluded from main savings calculations",
        "Reason": "Negative balance requires verification; original retained"
    })

for index in df.index[df["Savings_Outlier_Review"]]:
    cleaning_log.append({
        "Stage": "Financial review",
        "Customer_ID": df.at[index, "Customer_ID"],
        "Field": "Savings_Analysis",
        "Action": "Excluded from main analysis; included in sensitivity check",
        "Reason": "Extreme balance is unverified; original retained"
    })

for index in df.index[df["Flow_Mismatch_Flag"]]:
    cleaning_log.append({
        "Stage": "Financial review",
        "Customer_ID": df.at[index, "Customer_ID"],
        "Field": "Flow_Analysis",
        "Action": "Set analytical flow missing",
        "Reason": "Deposits minus withdrawals contradict recorded net flow"
    })

### Effect of the suspicious savings balance

Excluding the unverified balance reduces average savings from
approximately LKR 2.81 million to LKR 0.77 million.

Median savings remains approximately LKR 0.45 million in both cases.
This shows that the average is strongly influenced by one record.

The main savings analysis uses 489 valid balances. The remaining
11 customers stay in the dataset but are excluded from calculations
requiring valid savings.

The large balance requires source verification. Its exclusion is
an analytical assumption, and both scenarios are retained.

In [117]:
print("Total customers:", len(df))
print("Usable savings:", df["Savings_Analysis"].notna().sum())
print("Usable monthly flows:", df["Flow_Analysis"].notna().sum())
print("Contradictory flows:", df["Flow_Mismatch_Flag"].sum())

display(pd.DataFrame(financial_checks).round(2))

Total customers: 500
Usable savings: 489
Usable monthly flows: 492
Contradictory flows: 2


,Field,Missing,Negative,Zero,Minimum,Median,Maximum
0,Savings_Balance,6,4,0,-88000.00,442763.82,1.000000e+09
1,Monthly_Deposit_Avg,6,0,0,194.79,17858.54,3.471401e+05
2,Monthly_Withdrawal_Avg,0,0,0,100.99,9684.80,9.999999e+06
3,Net_Monthly_Flow,0,0,0,30.87,6044.87,1.962410e+05
4,Loan_Amount,283,0,0,54308.53,1421377.63,2.998363e+06
5,Outstanding_Loan_Balance,281,0,0,16258.84,669510.68,2.460698e+06
6,FD_Amount,355,0,0,113217.78,1101160.81,1.977886e+06


In [118]:
df["Flow_Group"] = "Unknown"

df.loc[df["Flow_Analysis"] > 0, "Flow_Group"] = "Growing"
df.loc[df["Flow_Analysis"] < 0, "Flow_Group"] = "Draining"
df.loc[df["Flow_Analysis"] == 0, "Flow_Group"] = "Stable"

flow_counts = df["Flow_Group"].value_counts().reindex(
    ["Growing", "Draining", "Stable", "Unknown"],
    fill_value=0
)

display(flow_counts.to_frame("Customers"))

,Customers
Flow_Group,
Growing,492
Draining,0
Stable,0
Unknown,8


## Age and date checks

The dataset does not specify a confirmed reporting date.
Today's date will not be used to recalculate historical customer ages.

Dates and recorded ages will be inspected before selecting and
documenting a reporting-date assumption.

In [119]:
date_columns = [
    "Date_of_Birth",
    "Account_Open_Date",
    "Loan_Start_Date",
    "Last_Login_Date",
    "KYC_Last_Updated"
]

date_checks = []

for column in date_columns:
    dates = pd.to_datetime(df[column], errors="coerce")

    date_checks.append({
        "Field": column,
        "Earliest": dates.min(),
        "Latest": dates.max(),
        "Missing": dates.isna().sum(),
        "Unparseable": (df[column].notna() & dates.isna()).sum()
    })

display(pd.DataFrame(date_checks))

,Field,Earliest,Latest,Missing,Unparseable
0,Date_of_Birth,1970-01-18,2020-05-01,1,0
1,Account_Open_Date,2015-01-14,2026-06-01,0,0
2,Loan_Start_Date,2008-03-20,2023-12-31,280,0
3,Last_Login_Date,2023-06-01,2027-01-01,169,0
4,KYC_Last_Updated,2020-01-07,2024-01-01,0,0


In [120]:
birth_date = pd.to_datetime(df["Date_of_Birth"], errors="coerce")
open_date = pd.to_datetime(df["Account_Open_Date"], errors="coerce")

age_review_mask = (
    df["Age"].isna()
    | (df["Age"] < 18)
    | (df["Age"] > 100)
    | (birth_date > open_date)
)

display(
    df.loc[
        age_review_mask,
        [
            "Customer_ID",
            "Age",
            "Date_of_Birth",
            "Account_Open_Date",
            "Age_DOB_Conflict"
        ]
    ]
)

,Customer_ID,Age,Date_of_Birth,Account_Open_Date,Age_DOB_Conflict
33,FSL-0034,4.0,2020-05-01,2021-09-07,False
88,FSL-0089,NaN,NaT,2016-09-28,True
270,FSL-0271,150.0,1975-03-11,2023-06-13,False


In [121]:
candidate_date = pd.Timestamp("2024-01-01")

birthday_not_reached = (
    (birth_date.dt.month > candidate_date.month)
    | (
        (birth_date.dt.month == candidate_date.month)
        & (birth_date.dt.day > candidate_date.day)
    )
)

age_at_candidate = (
    candidate_date.year
    - birth_date.dt.year
    - birthday_not_reached.astype(int)
).where(birth_date.notna())

comparable = df["Age"].notna() & age_at_candidate.notna()

print("Candidate date:", candidate_date.date())
print("Comparable age records:", comparable.sum())

print(
    "Recorded age matches candidate date:",
    (df.loc[comparable, "Age"] == age_at_candidate[comparable]).sum()
)

Candidate date: 2024-01-01
Comparable age records: 499
Recorded age matches candidate date: 481


In [122]:
age_mismatch = (
    comparable
    & df["Age"].ne(age_at_candidate)
)

age_review = df.loc[
    age_mismatch,
    ["Customer_ID", "Age", "Date_of_Birth", "Account_Open_Date"]
].copy()

age_review["Age_At_2024_01_01"] = age_at_candidate[age_mismatch]

age_review["Age_Difference"] = (
    age_review["Age"] - age_review["Age_At_2024_01_01"]
)

display(age_review)

,Customer_ID,Age,Date_of_Birth,Account_Open_Date,Age_At_2024_01_01,Age_Difference
33,FSL-0034,4.0,2020-05-01,2021-09-07,3.0,1.0
60,FSL-0061,99.0,1974-07-06,2016-10-08,49.0,50.0
66,FSL-0067,51.0,1973-01-07,2022-11-20,50.0,1.0
144,FSL-0145,53.0,1971-01-14,2022-11-10,52.0,1.0
179,FSL-0180,26.0,1998-01-02,2015-11-11,25.0,1.0
181,FSL-0182,39.0,1985-01-10,2021-07-20,38.0,1.0
183,FSL-0184,46.0,1978-01-02,2017-06-13,45.0,1.0
219,FSL-0220,50.0,1974-01-08,2018-07-07,49.0,1.0
270,FSL-0271,150.0,1975-03-11,2023-06-13,48.0,102.0
332,FSL-0333,51.0,1973-01-12,2016-01-31,50.0,1.0


In [123]:
display(flow_counts.to_frame("Customers"))

display(pd.DataFrame(date_checks))

display(
    df.loc[
        age_review_mask,
        [
            "Customer_ID",
            "Age",
            "Date_of_Birth",
            "Account_Open_Date",
            "Age_DOB_Conflict"
        ]
    ]
)

,Customers
Flow_Group,
Growing,492
Draining,0
Stable,0
Unknown,8


,Field,Earliest,Latest,Missing,Unparseable
0,Date_of_Birth,1970-01-18,2020-05-01,1,0
1,Account_Open_Date,2015-01-14,2026-06-01,0,0
2,Loan_Start_Date,2008-03-20,2023-12-31,280,0
3,Last_Login_Date,2023-06-01,2027-01-01,169,0
4,KYC_Last_Updated,2020-01-07,2024-01-01,0,0


,Customer_ID,Age,Date_of_Birth,Account_Open_Date,Age_DOB_Conflict
33,FSL-0034,4.0,2020-05-01,2021-09-07,False
88,FSL-0089,NaN,NaT,2016-09-28,True
270,FSL-0271,150.0,1975-03-11,2023-06-13,False


### Candidate reporting-date check

Using 1 January 2024 as a diagnostic date, 481 of 499 comparable
recorded ages match the age calculated from date of birth.

The agreement is approximately 96.4%, which supports an early-2024
reference period but does not establish an exact reporting date.

The 18 mismatches require review before age corrections are made.

### Savings-flow finding

All 492 customers with usable monthly-flow data have positive flow.
This represents 98.4% of the full customer base.

Eight customers have unknown analytical flow because of missing
deposit values or contradictory inputs.

No draining or stable customers were identified among usable records.
The result describes average flows, not observed month-by-month growth.
The uniformly positive pattern limits conclusions about financial stress.

In [124]:
df["Age_Analysis"] = df["Age"].copy()

df["Age_Invalid_Flag"] = (
    (df["Age"] < 0)
    | (df["Age"] > 100)
)

df.loc[df["Age_Invalid_Flag"], "Age_Analysis"] = float("nan")

df["Minor_Review_Flag"] = df["Age_Analysis"] < 18

print("Unavailable analytical ages:", df["Age_Analysis"].isna().sum())
print("Minor accounts requiring review:", df["Minor_Review_Flag"].sum())

Unavailable analytical ages: 2
Minor accounts requiring review: 1


In [125]:
candidate_date = pd.Timestamp("2024-01-01")

date_flag_summary = []

for column in [
    "Account_Open_Date",
    "Loan_Start_Date",
    "Last_Login_Date",
    "KYC_Last_Updated"
]:
    dates = pd.to_datetime(df[column], errors="coerce")

    flag_column = column + "_After_Candidate"
    df[flag_column] = dates > candidate_date

    date_flag_summary.append({
        "Field": column,
        "After_Candidate_Date": int(df[flag_column].sum())
    })

display(pd.DataFrame(date_flag_summary))

,Field,After_Candidate_Date
0,Account_Open_Date,2
1,Loan_Start_Date,0
2,Last_Login_Date,2
3,KYC_Last_Updated,0


In [126]:
print(age_review.to_string(index=False))

Customer_ID   Age Date_of_Birth Account_Open_Date  Age_At_2024_01_01  Age_Difference
   FSL-0034   4.0    2020-05-01        2021-09-07                3.0             1.0
   FSL-0061  99.0    1974-07-06        2016-10-08               49.0            50.0
   FSL-0067  51.0    1973-01-07        2022-11-20               50.0             1.0
   FSL-0145  53.0    1971-01-14        2022-11-10               52.0             1.0
   FSL-0180  26.0    1998-01-02        2015-11-11               25.0             1.0
   FSL-0182  39.0    1985-01-10        2021-07-20               38.0             1.0
   FSL-0184  46.0    1978-01-02        2017-06-13               45.0             1.0
   FSL-0220  50.0    1974-01-08        2018-07-07               49.0             1.0
   FSL-0271 150.0    1975-03-11        2023-06-13               48.0           102.0
   FSL-0333  51.0    1973-01-12        2016-01-31               50.0             1.0
   FSL-0341  35.0    1989-01-05        2023-05-13               3

In [127]:
cleaning_log = [
    entry for entry in cleaning_log
    if entry.get("Stage") != "Age review"
]

cleaning_log.append({
    "Stage": "Age review",
    "Customer_ID": "FSL-0271",
    "Field": "Age_Analysis",
    "Action": "Set analytical age missing; original retained",
    "Reason": "Recorded age 150 is unreliable; birth date not verified"
})

In [128]:
# Flag differences larger than one year against the diagnostic date.
df["Age_Major_Conflict"] = (
    comparable
    & df["Age"].sub(age_at_candidate).abs().gt(1)
)

# Keep one-year differences visible.
df["Age_One_Year_Difference"] = (
    comparable
    & df["Age"].sub(age_at_candidate).abs().eq(1)
)

# Rebuild the analytical age from recorded values.
df["Age_Analysis"] = df["Age"].copy()

exclude_age = (
    df["Age_DOB_Conflict"]
    | df["Age_Invalid_Flag"]
    | df["Age_Major_Conflict"]
)

df.loc[exclude_age, "Age_Analysis"] = float("nan")

df["Minor_Review_Flag"] = df["Age_Analysis"] < 18

print("Available analytical ages:", df["Age_Analysis"].notna().sum())
print("Unavailable analytical ages:", df["Age_Analysis"].isna().sum())
print("One-year differences:", df["Age_One_Year_Difference"].sum())

Available analytical ages: 497
Unavailable analytical ages: 3
One-year differences: 16


In [129]:
cleaning_log = [
    entry for entry in cleaning_log
    if entry.get("Stage") != "Age review"
]

for customer_id in ["FSL-0061", "FSL-0271"]:
    cleaning_log.append({
        "Stage": "Age review",
        "Customer_ID": customer_id,
        "Field": "Age_Analysis",
        "Action": "Set analytical age missing; original retained",
        "Reason": (
            "Large disagreement between recorded age and birth date; "
            "neither value independently verified"
        )
    })

In [130]:
age_bins = [0, 18, 30, 45, 60, float("inf")]

age_labels = [
    "Under 18",
    "18–29",
    "30–44",
    "45–59",
    "60+"
]

df["Age_Group"] = pd.cut(
    df["Age_Analysis"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

display(
    df["Age_Group"]
    .value_counts(sort=False, dropna=False)
    .to_frame("Customers")
)

,Customers
Age_Group,
Under 18,1
18–29,153
30–44,213
45–59,130
60+,0
NaN,3


In [131]:
candidate_age = age_at_candidate.mask(exclude_age)

candidate_group = pd.cut(
    candidate_age,
    bins=age_bins,
    labels=age_labels,
    right=False
)

both_available = (
    df["Age_Group"].notna()
    & candidate_group.notna()
)

band_changed = (
    both_available
    & df["Age_Group"].ne(candidate_group)
)

band_review = df.loc[
    band_changed,
    ["Customer_ID", "Age_Analysis", "Age_Group", "Savings_Analysis"]
].copy()

band_review["Age_At_Candidate"] = candidate_age[band_changed]
band_review["Group_At_Candidate"] = candidate_group[band_changed]

display(band_review)

,Customer_ID,Age_Analysis,Age_Group,Savings_Analysis,Age_At_Candidate,Group_At_Candidate


### Age-analysis policy

Recorded ages are used where there is no major unresolved conflict.
Three customers have unavailable analytical ages.

Sixteen one-year differences are retained and flagged. Fifteen involve
early-January birthdays, which could reflect a later January reference
date. The reporting date remains unconfirmed.

The minor account is retained in an Under 18 group pending verification.

Age-group results are checked against ages calculated at 1 January 2024
to assess whether the date assumption changes group membership.

# Q1 — Savings and Customer Behaviour

Savings are compared by customer segment, province, location type
and age group.

Calculations use Savings_Analysis. Missing or excluded balances
are not treated as zero.

Each summary shows total customers, usable balances, total savings,
average savings and median savings.

In [132]:
def savings_summary(group_column):
    work = df.copy()

    # Display missing categories as Unknown.
    work["Group"] = (
        work[group_column]
        .astype("string")
        .fillna("Unknown")
    )

    summary = work.groupby("Group").agg(
        Customers=("Customer_ID", "size"),
        Valid_Balances=("Savings_Analysis", "count"),
        Total_Savings=(
            "Savings_Analysis",
            lambda values: values.sum(min_count=1)
        ),
        Average_Savings=("Savings_Analysis", "mean"),
        Median_Savings=("Savings_Analysis", "median")
    )

    summary["Unavailable_Balances"] = (
        summary["Customers"] - summary["Valid_Balances"]
    )

    return summary

In [133]:
segment_savings = savings_summary("Customer_Segment_Clean")
province_savings = savings_summary("Province_Analysis")
location_savings = savings_summary("Urban_Rural_Clean")

print("SAVINGS BY SEGMENT")
display(
    segment_savings
    .sort_values("Average_Savings", ascending=False)
    .round(2)
)

print("SAVINGS BY PROVINCE")
display(
    province_savings
    .sort_values("Average_Savings", ascending=False)
    .round(2)
)

print("SAVINGS BY LOCATION TYPE")
display(
    location_savings
    .sort_values("Average_Savings", ascending=False)
    .round(2)
)

SAVINGS BY SEGMENT


,Customers,Valid_Balances,Total_Savings,Average_Savings,Median_Savings,Unavailable_Balances
Group,,,,,,
Premium,91,90,2.390220e+08,2655799.81,2504759.74,1
Unknown,5,5,2.522162e+06,504432.44,299601.41,0
Regular,313,305,1.294512e+08,424430.07,448297.06,8
Starter,91,89,7.353902e+06,82628.11,85217.43,2


SAVINGS BY PROVINCE


,Customers,Valid_Balances,Total_Savings,Average_Savings,Median_Savings,Unavailable_Balances
Group,,,,,,
Eastern,50,49,47287147.86,965043.83,562847.61,1
Sabaragamuwa,56,54,49961946.06,925221.22,409666.60,2
Western,103,100,85412655.47,854126.55,436777.12,3
North Western,55,53,41647056.89,785793.53,532220.84,2
Central,47,45,33247390.54,738830.90,430134.97,2
Southern,80,80,52672913.44,658411.42,378636.02,0
Northern,59,58,36588183.43,630830.75,398645.90,1
Uva,50,50,31531923.10,630638.46,416563.69,0


SAVINGS BY LOCATION TYPE


,Customers,Valid_Balances,Total_Savings,Average_Savings,Median_Savings,Unavailable_Balances
Group,,,,,,
Urban,189,184,1.652539e+08,898119.09,512586.53,5
Rural,123,120,8.801617e+07,733468.06,429709.61,3
Suburban,188,185,1.250791e+08,676103.45,368051.95,3


In [134]:
age_savings = savings_summary("Age_Group")

age_order = [
    "Under 18",
    "18–29",
    "30–44",
    "45–59",
    "60+",
    "Unknown"
]

age_savings = age_savings.reindex(age_order)

# An empty age group has zero customers, but no average or median.
count_columns = [
    "Customers",
    "Valid_Balances",
    "Unavailable_Balances"
]

age_savings[count_columns] = (
    age_savings[count_columns].fillna(0).astype(int)
)

display(age_savings.round(2))

known_groups = age_savings.drop(index="Unknown")

highest_group = known_groups["Total_Savings"].idxmax()

print("Age group with highest known total savings:", highest_group)

,Customers,Valid_Balances,Total_Savings,Average_Savings,Median_Savings,Unavailable_Balances
Group,,,,,,
Under 18,1,1,5.127600e+05,512759.98,512759.98,0
18–29,153,149,1.251561e+08,839973.90,355209.41,4
30–44,213,210,1.638422e+08,780201.01,483015.70,3
45–59,130,126,8.793487e+07,697895.81,461261.36,4
60+,0,0,NaN,NaN,NaN,0
Unknown,3,3,9.032619e+05,301087.29,82776.58,0


Age group with highest known total savings: 30–44


In [135]:
for name, table in [
    ("Segment", segment_savings),
    ("Province", province_savings),
    ("Location", location_savings),
    ("Age", age_savings)
]:
    print(
        name,
        "| Customers:", int(table["Customers"].sum()),
        "| Valid balances:", int(table["Valid_Balances"].sum()),
        "| Total savings:", round(table["Total_Savings"].sum(), 2)
    )

Segment | Customers: 500 | Valid balances: 489 | Total savings: 378349216.79
Province | Customers: 500 | Valid balances: 489 | Total savings: 378349216.79
Location | Customers: 500 | Valid balances: 489 | Total savings: 378349216.79
Age | Customers: 500 | Valid balances: 489 | Total savings: 378349216.79


## Q1 findings

Premium customers account for 18.2% of customers and approximately
63.2% of usable savings. Both their average and median balances are
substantially higher than those of Regular and Starter customers.
The segment definitions should be checked because balances may
contribute to how customers are classified.

Eastern has the highest average savings, while Western holds the
highest total savings. The dataset represents eight provinces and
does not include North Central.

Urban customers have higher average and median savings than Rural
and Suburban customers. Differences in customer composition may
partly explain this pattern.

Customers aged 30–44 hold the highest total savings, approximately
LKR 163.84 million. They are also the largest age group.

All 492 customers with usable flow data have positive average monthly
flow. Eight customers have unknown flow. These averages do not
establish that balances increased in every individual month.

Results use 489 usable savings balances. The suspicious large balance
is excluded from the main analysis pending verification.

In [136]:
# Province comparison excluding the four conflicting location records.
province_without_conflicts = (
    df.loc[~df["District_Province_Conflict"]]
    .groupby("Province_Analysis")["Savings_Analysis"]
    .agg(["count", "mean", "median"])
)

province_sensitivity = (
    province_savings[["Average_Savings"]]
    .rename(columns={"Average_Savings": "Main_Average"})
    .join(
        province_without_conflicts["mean"]
        .rename("Average_Without_Conflicts")
    )
)

display(province_sensitivity.round(2))

# Revisit the earlier age-band sensitivity check.
print("Customers changing age group:", len(band_review))
display(band_review)

,Main_Average,Average_Without_Conflicts
Group,,
Central,738830.90,749496.39
Eastern,965043.83,965043.83
North Western,785793.53,785793.53
Northern,630830.75,627963.95
Sabaragamuwa,925221.22,925221.22
Southern,658411.42,665858.02
Uva,630638.46,630638.46
Western,854126.55,862324.62


Customers changing age group: 0


,Customer_ID,Age_Analysis,Age_Group,Savings_Analysis,Age_At_Candidate,Group_At_Candidate


# Q2 — Loan Portfolio and Risk

Loan penetration measures the percentage of customers classified
as borrowers among customers with known analytical loan status.

The analysis uses Has_Loan_Clean. Inferred borrowers are identified
separately, and unknown loan statuses are disclosed.

Original recorded flags are retained for comparison.

In [137]:
total_customers = len(df)

borrowers = df["Has_Loan_Clean"].eq("Yes").sum()
non_borrowers = df["Has_Loan_Clean"].eq("No").sum()
unknown_status = df["Has_Loan_Clean"].eq("Unknown").sum()

known_status = borrowers + non_borrowers

loan_penetration = (
    borrowers / known_status * 100
    if known_status > 0
    else float("nan")
)

print("Total customers:", total_customers)
print("Borrowers:", borrowers)
print("Non-borrowers:", non_borrowers)
print("Unknown loan status:", unknown_status)
print(f"Loan penetration among known statuses: {loan_penetration:.2f}%")

print(
    f"Classified borrowers as a share of all customers: "
    f"{borrowers / total_customers * 100:.2f}%"
)

Total customers: 500
Borrowers: 221
Non-borrowers: 274
Unknown loan status: 5
Loan penetration among known statuses: 44.65%
Classified borrowers as a share of all customers: 44.20%


In [138]:
def loan_penetration_summary(group_column):
    work = df.copy()

    work["Group"] = (
        work[group_column].astype("string").fillna("Unknown")
    )

    work["Borrower"] = work["Has_Loan_Clean"].eq("Yes")
    work["Known_Status"] = work["Has_Loan_Clean"].isin(["Yes", "No"])
    work["Unknown_Status"] = work["Has_Loan_Clean"].eq("Unknown")

    summary = work.groupby("Group").agg(
        Customers=("Customer_ID", "size"),
        Borrowers=("Borrower", "sum"),
        Known_Status=("Known_Status", "sum"),
        Unknown_Status=("Unknown_Status", "sum")
    )

    denominator = summary["Known_Status"].replace(0, float("nan"))

    summary["Penetration_Percent"] = (
        summary["Borrowers"] / denominator * 100
    )

    total_borrowers = summary["Borrowers"].sum()

    if total_borrowers > 0:
        summary["Share_of_All_Borrowers_Percent"] = (
            summary["Borrowers"] / total_borrowers * 100
        )
    else:
        summary["Share_of_All_Borrowers_Percent"] = float("nan")

    return summary.sort_values("Penetration_Percent", ascending=False)


segment_loans = loan_penetration_summary("Customer_Segment_Clean")
channel_loans = loan_penetration_summary("Acquisition_Channel_Clean")

print("LOANS BY SEGMENT")
display(segment_loans.round(2))

print("LOANS BY ACQUISITION CHANNEL")
display(channel_loans.round(2))

LOANS BY SEGMENT


,Customers,Borrowers,Known_Status,Unknown_Status,Penetration_Percent,Share_of_All_Borrowers_Percent
Group,,,,,,
Regular,313,142,311,2,45.66,64.25
Premium,91,41,90,1,45.56,18.55
Starter,91,36,89,2,40.45,16.29
Unknown,5,2,5,0,40.00,0.90


LOANS BY ACQUISITION CHANNEL


,Customers,Borrowers,Known_Status,Unknown_Status,Penetration_Percent,Share_of_All_Borrowers_Percent
Group,,,,,,
Agent,117,56,116,1,48.28,25.34
Referral,129,57,128,1,44.53,25.79
Branch,123,53,121,2,43.80,23.98
Digital,131,55,130,1,42.31,24.89


In [139]:
status_comparison = []

for column, label in [
    ("Has_Loan", "Original flags"),
    ("Has_Loan_Clean", "Analytical classifications")
]:
    known = df[column].isin(["Yes", "No"]).sum()
    yes = df[column].eq("Yes").sum()

    status_comparison.append({
        "Version": label,
        "Borrowers": yes,
        "Known_Status": known,
        "Unknown_Status": len(df) - known,
        "Penetration_Percent": (
            yes / known * 100 if known > 0 else float("nan")
        )
    })

display(pd.DataFrame(status_comparison).round(2))

,Version,Borrowers,Known_Status,Unknown_Status,Penetration_Percent
0,Original flags,218,495,5,44.04
1,Analytical classifications,221,495,5,44.65


### Q2: Loan penetration

There are 221 classified borrowers among 495 customers with known
analytical loan status, giving penetration of 44.65%.
Five customers have unknown status.

Regular customers represent 64.25% of borrowers. However, penetration
within Regular and Premium segments is almost identical.

Agent has the highest loan penetration at 48.28%, while Referral
has the highest borrower count at 57. Borrower counts are relatively
evenly distributed across acquisition channels.

Cleaning increases overall penetration by approximately 0.61
percentage points. Three borrower classifications are inferred
from positive outstanding balances.

In [140]:
is_borrower = df["Has_Loan_Clean"].eq("Yes")

valid_loan_balance = (
    df["Outstanding_Loan_Balance"].notna()
    & df["Outstanding_Loan_Balance"].ge(0)
)

valid_savings = (
    df["Savings_Analysis"].notna()
    & df["Savings_Analysis"].gt(0)
)

df["Debt_Savings_Ratio"] = float("nan")

ratio_available = (
    is_borrower
    & valid_loan_balance
    & valid_savings
)

df.loc[ratio_available, "Debt_Savings_Ratio"] = (
    df.loc[ratio_available, "Outstanding_Loan_Balance"]
    / df.loc[ratio_available, "Savings_Analysis"]
)

print("Classified borrowers:", is_borrower.sum())
print("Ratios available:", ratio_available.sum())
print("Ratios unavailable:", (is_borrower & ~ratio_available).sum())

Classified borrowers: 221
Ratios available: 215
Ratios unavailable: 6


In [141]:
above_one = ratio_available & df["Debt_Savings_Ratio"].gt(1)

above_one_count = int(above_one.sum())
available_count = int(ratio_available.sum())

above_one_exposure = df.loc[
    above_one,
    "Outstanding_Loan_Balance"
].sum(min_count=1)

above_one_percent = (
    above_one_count / available_count * 100
    if available_count > 0
    else float("nan")
)

print("Customers with ratio above 1.0:", above_one_count)
print(f"Share of assessable borrowers: {above_one_percent:.2f}%")
print(f"Outstanding loan exposure: LKR {above_one_exposure:,.2f}")

Customers with ratio above 1.0: 128
Share of assessable borrowers: 59.53%
Outstanding loan exposure: LKR 131,767,566.49


In [142]:
df["Ratio_Assessment"] = "Not classified as borrower"

df.loc[is_borrower, "Ratio_Assessment"] = "Available"

df.loc[
    is_borrower & ~valid_loan_balance & valid_savings,
    "Ratio_Assessment"
] = "Loan balance unavailable or invalid"

df.loc[
    is_borrower & valid_loan_balance & ~valid_savings,
    "Ratio_Assessment"
] = "Savings unavailable or nonpositive"

df.loc[
    is_borrower & ~valid_loan_balance & ~valid_savings,
    "Ratio_Assessment"
] = "Both inputs unavailable or invalid"

ratio_coverage = (
    df.loc[is_borrower]
    .groupby("Ratio_Assessment")
    .agg(
        Borrowers=("Customer_ID", "size"),
        Recorded_Balances=("Outstanding_Loan_Balance", "count"),
        Recorded_Outstanding=(
            "Outstanding_Loan_Balance",
            lambda values: values.sum(min_count=1)
        )
    )
)

display(ratio_coverage.round(2))

,Borrowers,Recorded_Balances,Recorded_Outstanding
Ratio_Assessment,,,
Available,215,215,1.639242e+08
Loan balance unavailable or invalid,2,0,NaN
Savings unavailable or nonpositive,4,4,4.081197e+06


In [143]:
original_savings_usable = (
    df["Savings_Balance"].notna()
    & df["Savings_Balance"].gt(0)
)

alternative_available = (
    is_borrower
    & valid_loan_balance
    & original_savings_usable
)

alternative_ratio = (
    df.loc[alternative_available, "Outstanding_Loan_Balance"]
    / df.loc[alternative_available, "Savings_Balance"]
)

alternative_above_one = alternative_ratio.gt(1)

print(
    "Ratios available including suspicious savings:",
    len(alternative_ratio)
)

print(
    "Customers above 1.0 including suspicious savings:",
    alternative_above_one.sum()
)

alternative_exposure = df.loc[
    alternative_ratio.index[alternative_above_one],
    "Outstanding_Loan_Balance"
].sum(min_count=1)

print(f"Alternative exposure: LKR {alternative_exposure:,.2f}")

Ratios available including suspicious savings: 215
Customers above 1.0 including suspicious savings: 128
Alternative exposure: LKR 131,767,566.49


### Q2: Debt-to-savings exposure

Debt-to-savings ratios could be calculated for 215 of 221 classified
borrowers. Of these, 128 borrowers (59.53%) have ratios above 1.0.

Their combined outstanding loan balance is LKR 131,767,566.49.
This is outstanding exposure, not expected credit loss.

Six borrowers could not be assessed. Four have unusable savings
and approximately LKR 4.08 million in recorded outstanding loans.
Two have missing outstanding balances, so total portfolio exposure
cannot be fully measured.

Including the suspicious savings balance does not change this result.

The ratio considers only savings held with this company. It does
not account for income, external assets, collateral or other debts.

In [144]:
df["Repayment_Status_Clean"] = (
    df["Loan_Repayment_Status"]
    .astype("string")
    .str.strip()
    .str.title()
)

display(
    df.loc[
        df["Has_Loan_Clean"].eq("Yes"),
        "Repayment_Status_Clean"
    ].value_counts(dropna=False).to_frame("Borrowers")
)

,Borrowers
Repayment_Status_Clean,
On-Time,133
Delayed,63
Defaulted,23
<NA>,2


In [145]:
borrower_data = df.loc[
    df["Has_Loan_Clean"].eq("Yes")
].copy()

valid_statuses = ["On-Time", "Delayed", "Defaulted"]

borrower_data["Status_Known"] = (
    borrower_data["Repayment_Status_Clean"].isin(valid_statuses)
)

borrower_data["Is_Defaulted"] = (
    borrower_data["Repayment_Status_Clean"]
    .eq("Defaulted")
    .fillna(False)
)

borrower_data["Is_Delayed"] = (
    borrower_data["Repayment_Status_Clean"]
    .eq("Delayed")
    .fillna(False)
)


def default_summary(group_columns):
    work = borrower_data.copy()

    for column in group_columns:
        work[column] = (
            work[column].astype("string").fillna("Unknown")
        )

    result = work.groupby(group_columns).agg(
        Borrowers=("Customer_ID", "size"),
        Known_Status=("Status_Known", "sum"),
        Defaulted=("Is_Defaulted", "sum"),
        Delayed=("Is_Delayed", "sum")
    )

    result["Unknown_Status"] = (
        result["Borrowers"] - result["Known_Status"]
    )

    denominator = result["Known_Status"].replace(0, float("nan"))

    result["Default_Rate_Percent"] = (
        result["Defaulted"] / denominator * 100
    )

    result["Delayed_Rate_Percent"] = (
        result["Delayed"] / denominator * 100
    )

    return result.sort_values(
        ["Default_Rate_Percent", "Known_Status"],
        ascending=[False, False]
    )

In [146]:
known_count = int(borrower_data["Status_Known"].sum())
default_count = int(borrower_data["Is_Defaulted"].sum())

overall_default_rate = (
    default_count / known_count * 100
    if known_count > 0
    else float("nan")
)

print("Borrowers with known repayment status:", known_count)
print("Defaulted borrowers:", default_count)
print(
    "Unknown repayment status:",
    len(borrower_data) - known_count
)
print(f"Overall observed default rate: {overall_default_rate:.2f}%")

segment_defaults = default_summary(["Customer_Segment_Clean"])
district_defaults = default_summary(["District_Clean"])
loan_type_defaults = default_summary(["Loan_Type"])

print("\nBY SEGMENT")
display(segment_defaults.round(2))

print("\nBY DISTRICT")
display(district_defaults.round(2))

print("\nBY LOAN TYPE")
display(loan_type_defaults.round(2))

Borrowers with known repayment status: 219
Defaulted borrowers: 23
Unknown repayment status: 2
Overall observed default rate: 10.50%

BY SEGMENT


,Borrowers,Known_Status,Defaulted,Delayed,Unknown_Status,Default_Rate_Percent,Delayed_Rate_Percent
Customer_Segment_Clean,,,,,,,
Starter,36,36,7,8,0,19.44,22.22
Premium,41,41,4,11,0,9.76,26.83
Regular,142,140,12,44,2,8.57,31.43
Unknown,2,2,0,0,0,0.0,0.0



BY DISTRICT


,Borrowers,Known_Status,Defaulted,Delayed,Unknown_Status,Default_Rate_Percent,Delayed_Rate_Percent
District_Clean,,,,,,,
Kurunegala,24,24,4,7,0,16.67,29.17
Colombo,25,25,4,8,0,16.0,32.0
Ratnapura,27,27,4,9,0,14.81,33.33
Gampaha,25,25,3,6,0,12.0,24.0
Matara,17,17,2,3,0,11.76,17.65
Kandy,20,19,2,3,1,10.53,15.79
Galle,15,15,1,3,0,6.67,20.0
Trincomalee,19,18,1,4,1,5.56,22.22
Badulla,23,23,1,11,0,4.35,47.83



BY LOAN TYPE


,Borrowers,Known_Status,Defaulted,Delayed,Unknown_Status,Default_Rate_Percent,Delayed_Rate_Percent
Loan_Type,,,,,,,
Unknown,3,1,1,0,2,100.0,0.0
Vehicle,51,51,8,13,0,15.69,25.49
Housing,53,53,6,14,0,11.32,26.42
Personal,55,55,4,17,0,7.27,30.91
Business,59,59,4,19,0,6.78,32.2


In [147]:
combination_defaults = default_summary([
    "Customer_Segment_Clean",
    "Loan_Type"
])

combination_defaults["Small_Group_Flag"] = (
    combination_defaults["Known_Status"] < 10
)

combination_defaults["Above_Overall_Rate"] = (
    combination_defaults["Default_Rate_Percent"]
    > overall_default_rate
)

display(combination_defaults.round(2))

Borrowers  Known_Status  Defaulted  Delayed  \
Customer_Segment_Clean Loan_Type                                                
Starter                Unknown            1             1          1        0   
                       Vehicle           13            13          4        4   
Premium                Business          13            13          2        3   
                       Vehicle            7             7          1        1   
Regular                Housing           31            31          4       10   
Starter                Housing            8             8          1        0   
                       Business           9             9          1        3   
Regular                Vehicle           31            31          3        8   
                       Personal          42            42          4       13   
Premium                Housing           13            13          1        4   
Regular                Business          36            36          1       13   
Premium                Personal           8             8          0        3   
Starter                Personal           5             5          0        1   
Unknown                Business           1             1          0        0   
                       Housing            1             1          0        0   
Regular                Unknown            2             0          0        0   

                                  Unknown_Status  Default_Rate_Percent  \
Customer_Segment_Clean Loan_Type                                         
Starter                Unknown                 0                 100.0   
                       Vehicle                 0                 30.77   
Premium                Business                0                 15.38   
                       Vehicle                 0                 14.29   
Regular                Housing                 0                  12.9   
Starter                Housing                 0                  12.5   
                       Business                0                 11.11   
Regular                Vehicle                 0                  9.68   
                       Personal                0                  9.52   
Premium                Housing                 0                  7.69   
Regular                Business                0                  2.78   
Premium                Personal                0                   0.0   
Starter                Personal                0                   0.0   
Unknown                Business                0                   0.0   
                       Housing                 0                   0.0   
Regular                Unknown                 2                  <NA>   

                                  Delayed_Rate_Percent  Small_Group_Flag  \
Customer_Segment_Clean Loan_Type                                           
Starter                Unknown                     0.0              True   
                       Vehicle                   30.77             False   
Premium                Business                  23.08             False   
                       Vehicle                   14.29              True   
Regular                Housing                   32.26             False   
Starter                Housing                     0.0              True   
                       Business                  33.33              True   
Regular                Vehicle                   25.81             False   
                       Personal                  30.95             False   
Premium                Housing                   30.77             False   
Regular                Business                  36.11             False   
Premium                Personal                   37.5              True   
Starter                Personal                   20.0              True   
Unknown                Business                    0.0              True   
                       Hou

### Q2: Repayment findings

The observed default rate is 10.50%, based on 23 defaults among
219 borrowers with known repayment status. Two borrowers have
unknown repayment status.

Starter customers have the highest segment default rate at 19.44%.
Regular customers have the largest default count, with 12 cases.

Vehicle loans have the highest default rate among known loan types,
at 15.69%. The Starter–Vehicle combination has four defaults among
13 borrowers, giving a rate of 30.77%.

These are descriptive signals for investigation. Small group sizes,
loan age, customer characteristics and missing repayment history
limit conclusions about the causes of default.

Unknown loan type is treated as a data-quality category.
Delayed repayments are reported separately from defaults.

# Q3 — Digital Engagement and Product Holdings

App users and non-users are compared using valid savings balances.
Both averages and medians are reported.

Comparisons are repeated within customer segments because segment
composition may explain an overall difference.

In [148]:
app_savings = savings_summary("Mobile_App_User")

display(app_savings.round(2))

app_average = app_savings.loc["Yes", "Average_Savings"]
non_app_average = app_savings.loc["No", "Average_Savings"]

difference = app_average - non_app_average

percentage_difference = (
    difference / non_app_average * 100
    if non_app_average > 0
    else float("nan")
)

print(f"Average difference, app minus non-app: LKR {difference:,.2f}")
print(f"Difference relative to non-app average: {percentage_difference:.2f}%")

,Customers,Valid_Balances,Total_Savings,Average_Savings,Median_Savings,Unavailable_Balances
Group,,,,,,
No,168,164,1.308930e+08,798127.91,444521.96,4
Yes,332,325,2.474562e+08,761403.81,448866.68,7


Average difference, app minus non-app: LKR -36,724.10
Difference relative to non-app average: -4.60%


In [149]:
app_work = df.copy()

app_work["Segment_Group"] = (
    app_work["Customer_Segment_Clean"]
    .astype("string")
    .fillna("Unknown")
)

app_by_segment = app_work.groupby(
    ["Segment_Group", "Mobile_App_User"]
).agg(
    Customers=("Customer_ID", "size"),
    Valid_Balances=("Savings_Analysis", "count"),
    Average_Savings=("Savings_Analysis", "mean"),
    Median_Savings=("Savings_Analysis", "median")
)

display(app_by_segment.round(2))

Customers  Valid_Balances  Average_Savings  \
Segment_Group Mobile_App_User                                               
Premium       No                      36              36       2367843.70   
              Yes                     55              54       2847770.55   
Regular       No                     102             100        433754.63   
              Yes                    211             205        419881.50   
Starter       No                      29              27         76867.10   
              Yes                     62              62         85136.93   
Unknown       No                       1               1        199730.37   
              Yes                      4               4        580607.95   

                               Median_Savings  
Segment_Group Mobile_App_User                  
Premium       No                   2109862.23  
              Yes                  2713031.22  
Regular       No                    422737.57  
              Yes                   463938.89  
Starter       No                     84415.13  
              Yes                    88479.22  
Unknown       No                    199730.37  
              Yes                   418153.84

In [150]:
cross_sell_mask = (
    df["Savings_Analysis"].gt(0)
    & df["Has_Fixed_Deposit"].eq("No")
    & df["Has_Insurance"].eq("No")
)

cross_sell_count = int(cross_sell_mask.sum())

cross_sell_savings = df.loc[
    cross_sell_mask, "Savings_Analysis"
].sum(min_count=1)

# Customers whose product flags qualify but savings cannot be assessed.
product_flags_qualify = (
    df["Has_Fixed_Deposit"].eq("No")
    & df["Has_Insurance"].eq("No")
)

unassessed_cross_sell = (
    product_flags_qualify
    & df["Savings_Analysis"].isna()
).sum()

print("Potential cross-sell customers:", cross_sell_count)
print(f"Savings held by this group: LKR {cross_sell_savings:,.2f}")
print("Additional customers with unusable savings:", unassessed_cross_sell)

Potential cross-sell customers: 251
Savings held by this group: LKR 222,385,271.58
Additional customers with unusable savings: 6


In [151]:
channel_savings = savings_summary("Acquisition_Channel_Clean")

channel_defaults = default_summary(["Acquisition_Channel_Clean"])

channel_comparison = channel_savings[
    ["Customers", "Valid_Balances", "Average_Savings", "Median_Savings"]
].join(
    channel_defaults[
        ["Borrowers", "Known_Status", "Defaulted", "Default_Rate_Percent"]
    ]
)

display(
    channel_comparison
    .sort_values("Average_Savings", ascending=False)
    .round(2)
)

,Customers,Valid_Balances,Average_Savings,Median_Savings,Borrowers,Known_Status,Defaulted,Default_Rate_Percent
Group,,,,,,,,
Branch,123,118,841984.38,404524.36,53,53,6,11.32
Digital,131,129,830903.09,482677.41,55,55,7,12.73
Referral,129,126,768937.97,454413.98,57,55,7,12.73
Agent,117,116,645882.56,418861.29,56,56,3,5.36


## Q3 findings

The association between app use and savings differs by segment.
Premium and Starter app users have higher average and median savings.
Regular app users have a slightly lower average but a higher median.
These comparisons do not establish that app use causes higher savings.

The potential cross-sell pool contains 251 customers holding
LKR 222,385,271.58 in usable savings. They have no recorded fixed
deposit or insurance. Six additional customers have qualifying
product flags but unusable savings values.

Branch has the highest average savings, while Agent has the lowest
observed default rate. No acquisition channel leads on both measures.

The snapshot cannot establish consistent channel performance over time.
Acquisition costs, customer mix and loan age should be examined before
reallocating acquisition spending.

In [152]:
# Growth
valid_flow_count = df["Flow_Analysis"].notna().sum()
growing_count = df["Flow_Analysis"].gt(0).sum()

# Lending
known_loan_count = df["Has_Loan_Clean"].isin(["Yes", "No"]).sum()
borrower_count = df["Has_Loan_Clean"].eq("Yes").sum()

# Risk
known_repayment_count = borrower_data["Status_Known"].sum()
defaulted_count = borrower_data["Is_Defaulted"].sum()

# Digital adoption
known_app_count = df["Mobile_App_User"].isin(["Yes", "No"]).sum()
app_user_count = df["Mobile_App_User"].eq("Yes").sum()

# Compliance: unknown statuses remain in the denominator.
verified_kyc_count = df["KYC_Status_Clean"].eq("Verified").sum()


def percentage(numerator, denominator):
    if denominator == 0:
        return float("nan")
    return numerator / denominator * 100


kpi_table = pd.DataFrame([
    {
        "KPI": "Savings-growing customer rate",
        "Area": "Growth",
        "Numerator": growing_count,
        "Denominator": valid_flow_count
    },
    {
        "KPI": "Loan penetration",
        "Area": "Lending",
        "Numerator": borrower_count,
        "Denominator": known_loan_count
    },
    {
        "KPI": "Observed borrower default rate",
        "Area": "Risk",
        "Numerator": defaulted_count,
        "Denominator": known_repayment_count
    },
    {
        "KPI": "Mobile app adoption",
        "Area": "Engagement",
        "Numerator": app_user_count,
        "Denominator": known_app_count
    },
    {
        "KPI": "KYC verified coverage",
        "Area": "Compliance",
        "Numerator": verified_kyc_count,
        "Denominator": len(df)
    }
])

kpi_table["Current_Percent"] = kpi_table.apply(
    lambda row: percentage(row["Numerator"], row["Denominator"]),
    axis=1
)

display(kpi_table.round(2))

,KPI,Area,Numerator,Denominator,Current_Percent
0,Savings-growing customer rate,Growth,492,492,100.00
1,Loan penetration,Lending,221,495,44.65
2,Observed borrower default rate,Risk,23,219,10.50
3,Mobile app adoption,Engagement,332,500,66.40
4,KYC verified coverage,Compliance,351,500,70.20


In [153]:
kpi_details = {
    "Savings-growing customer rate": {
        "Definition": "Share of customers with usable flow whose flow is positive",
        "Formula": "Positive Flow_Analysis / non-missing Flow_Analysis * 100",
        "Business_Use": "Monitor savings behaviour and investigate flow data gaps",
        "Proposed_Target": "Maintain validated baseline; resolve 8 unavailable flows",
        "Target_Reason": "Reported rate is already 100%; validate before setting growth goals"
    },

    "Loan penetration": {
        "Definition": "Share of customers with known loan status classified as borrowers",
        "Formula": "Has_Loan_Clean Yes / Has_Loan_Clean Yes or No * 100",
        "Business_Use": "Balance lending reach with repayment risk",
        "Proposed_Target": "Maintain approximately 44-45% initially",
        "Target_Reason": "Understand current repayment problems before targeting expansion"
    },

    "Observed borrower default rate": {
        "Definition": "Share of borrowers with known repayment status marked Defaulted",
        "Formula": "Defaulted borrowers / borrowers with recognised repayment status * 100",
        "Business_Use": "Prioritise collections and investigate risky customer groups",
        "Proposed_Target": "9.50% or lower",
        "Target_Reason": "Proposed 1 percentage-point reduction; also monitor existing borrowers"
    },

    "Mobile app adoption": {
        "Definition": "Share of customers with known app status recorded as app users",
        "Formula": "Mobile_App_User Yes / Mobile_App_User Yes or No * 100",
        "Business_Use": "Evaluate app onboarding efforts",
        "Proposed_Target": "70.00%",
        "Target_Reason": "Requires 18 additional users if the customer base stays at 500"
    },

    "KYC verified coverage": {
        "Definition": "Share of all customers recorded as KYC Verified",
        "Formula": "KYC_Status_Clean Verified / all unique customers * 100",
        "Business_Use": "Prioritise verification backlog and missing-status reviews",
        "Proposed_Target": "80.00% initial milestone",
        "Target_Reason": "Requires 49 additional verified records at a fixed base of 500"
    }
}

for field in [
    "Definition",
    "Formula",
    "Business_Use",
    "Proposed_Target",
    "Target_Reason"
]:
    kpi_table[field] = kpi_table["KPI"].map(
        lambda name: kpi_details[name][field]
    )

display(
    kpi_table[
        ["KPI", "Current_Percent", "Proposed_Target"]
    ].round(2)
)

,KPI,Current_Percent,Proposed_Target
0,Savings-growing customer rate,100.00,Maintain validated baseline; resolve 8 unavail...
1,Loan penetration,44.65,Maintain approximately 44-45% initially
2,Observed borrower default rate,10.50,9.50% or lower
3,Mobile app adoption,66.40,70.00%
4,KYC verified coverage,70.20,80.00% initial milestone


### KPI targets and monitoring

Targets are proposed internal pilot goals, not external benchmarks.
The evaluation horizon is three months after a verified baseline
is established.

Targets require validation against operational capacity and business
priorities. Customer-count examples assume a fixed population of 500.

This dataset provides a baseline snapshot. Monthly monitoring requires
new snapshots with consistent definitions and reporting dates.

App adoption does not measure recent activity. Recorded KYC verification
does not independently establish current compliance.

Default counts, outstanding defaulted balances and outcomes for existing
borrowers should accompany the default rate to prevent misleading
improvements caused only by changes in the denominator.

In [154]:
dashboard_data_folder = project_folder / "data" / "dashboard"
dashboard_data_folder.mkdir(parents=True, exist_ok=True)

kpi_export = kpi_table.copy()

# Power BI percentage formatting expects 0.664 for 66.4%.
kpi_export["Current_Rate"] = (
    kpi_export["Current_Percent"] / 100
)

kpi_path = dashboard_data_folder / "kpi_summary.csv"

kpi_export.to_csv(kpi_path, index=False)

print("Saved:", kpi_path)

Saved: c:\Users\Tharindu\Desktop\FinSight-Customer-Analytics\data\dashboard\kpi_summary.csv


In [155]:
kpi_path = project_folder / "data" / "dashboard" / "kpi_summary.csv"

print("Dashboard file exists:", kpi_path.exists())

display(
    kpi_table[
        ["KPI", "Numerator", "Denominator", "Current_Percent"]
    ].round(2)
)

Dashboard file exists: True


,KPI,Numerator,Denominator,Current_Percent
0,Savings-growing customer rate,492,492,100.00
1,Loan penetration,221,495,44.65
2,Observed borrower default rate,23,219,10.50
3,Mobile app adoption,332,500,66.40
4,KYC verified coverage,351,500,70.20


In [156]:
dashboard_folder = project_folder / "data" / "dashboard"
dashboard_folder.mkdir(parents=True, exist_ok=True)

segment_chart = (
    segment_savings
    .rename_axis("Segment")
    .reset_index()
)

output_file = dashboard_folder / "savings_by_segment.csv"
segment_chart.to_csv(output_file, index=False)

print("Saved:", output_file)
display(segment_chart)

Saved: c:\Users\Tharindu\Desktop\FinSight-Customer-Analytics\data\dashboard\savings_by_segment.csv


,Segment,Customers,Valid_Balances,Total_Savings,Average_Savings,Median_Savings,Unavailable_Balances
0,Premium,91,90,2.390220e+08,2.655800e+06,2504759.735,1
1,Regular,313,305,1.294512e+08,4.244301e+05,448297.060,8
2,Starter,91,89,7.353902e+06,8.262811e+04,85217.430,2
3,Unknown,5,5,2.522162e+06,5.044324e+05,299601.410,0


In [157]:
# Prepare the existing default summary for Power BI.
default_chart = (
    segment_defaults
    .rename_axis("Segment")
    .reset_index()
)

# Store the rate as a decimal for percentage formatting.
default_chart["Default_Rate"] = (
    default_chart["Defaulted"]
    / default_chart["Known_Status"].replace(0, float("nan"))
)

# Export the summary.
output_file = dashboard_folder / "defaults_by_segment.csv"
default_chart.to_csv(output_file, index=False)

print("Saved:", output_file)

display(default_chart[
    ["Segment", "Borrowers", "Known_Status",
     "Defaulted", "Unknown_Status", "Default_Rate"]
])

Saved: c:\Users\Tharindu\Desktop\FinSight-Customer-Analytics\data\dashboard\defaults_by_segment.csv


,Segment,Borrowers,Known_Status,Defaulted,Unknown_Status,Default_Rate
0,Starter,36,36,7,0,0.194444
1,Premium,41,41,4,0,0.097561
2,Regular,142,140,12,2,0.085714
3,Unknown,2,2,0,0,0.0


In [158]:
# Prepare the channel comparison already calculated.
channel_chart = (
    channel_comparison
    .rename_axis("Channel")
    .reset_index()
)

# Export for Power BI.
output_file = dashboard_folder / "channel_comparison.csv"
channel_chart.to_csv(output_file, index=False)

print("Saved:", output_file)
display(channel_chart)

Saved: c:\Users\Tharindu\Desktop\FinSight-Customer-Analytics\data\dashboard\channel_comparison.csv


,Channel,Customers,Valid_Balances,Average_Savings,Median_Savings,Borrowers,Known_Status,Defaulted,Default_Rate_Percent
0,Agent,117,116,645882.561207,418861.290,56,56,3,5.357143
1,Branch,123,118,841984.378136,404524.360,53,53,6,11.320755
2,Digital,131,129,830903.093101,482677.410,55,55,7,12.727273
3,Referral,129,126,768937.968730,454413.985,57,55,7,12.727273


In [159]:
# Summarise the cross-sell group identified earlier.
cross_sell_summary = pd.DataFrame({
    "Potential_Customers": [int(cross_sell_mask.sum())],
    "Savings_Held": [
        df.loc[cross_sell_mask, "Savings_Analysis"].sum()
    ]
})

output_file = dashboard_folder / "cross_sell_summary.csv"
cross_sell_summary.to_csv(output_file, index=False)

print("Saved:", output_file)
display(cross_sell_summary)

Saved: c:\Users\Tharindu\Desktop\FinSight-Customer-Analytics\data\dashboard\cross_sell_summary.csv


,Potential_Customers,Savings_Held
0,251,2.223853e+08


In [160]:
from pathlib import Path
import pandas as pd

csv_path = Path(
    r"C:\Users\Tharindu\Desktop\FinSight-Customer-Analytics"
    r"\data\dashboard\kpi_summary.csv"
)

kpi_export = pd.read_csv(csv_path)

formulas = {
    "Savings-growing customer rate":
        "(Customers with positive usable net flow / customers with usable flow data) × 100",

    "Loan penetration":
        "(Borrowers / customers with known loan status) × 100",

    "Observed borrower default rate":
        "(Defaulted borrowers / borrowers with known repayment status) × 100",

    "KYC verified coverage":
        "(Customers marked verified / all customers) × 100",

    "Mobile app adoption":
        "(App users / customers with known app-use status) × 100",
}

for kpi, formula in formulas.items():
    kpi_export.loc[kpi_export["KPI"] == kpi, "Formula"] = formula

kpi_export.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Updated KPI formulas successfully.")

Updated KPI formulas successfully.


## Q5 — Business Findings and Recommendations

### Main findings

1. **Savings are concentrated in Premium customers.**
   Premium customers represent 91 of 500 customers (18.2%) but hold
   approximately 63.2% of usable savings balances.
   Their average savings are approximately LKR 2.66 million.
   This suggests that retaining these customers matters to the savings base.
   However, the segment may itself be defined partly by savings balance.

2. **Starter customers have the highest observed segment default rate.**
   Seven of 36 Starter borrowers with known repayment status are marked
   Defaulted (19.44%), compared with 23 of 219 borrowers overall (10.50%).
   This is a signal for further investigation, not proof that every
   Starter customer is high risk.

3. **There is a potential opportunity to offer additional products.**
   There are 251 customers with positive usable savings who have neither
   a fixed deposit nor insurance in the dataset.
   Together, they hold LKR 222.39 million in savings.
   These are potential customers to assess, not confirmed buyers.
   The savings balance is not expected revenue.

4. **No acquisition channel leads on both savings and defaults.**
   Branch customers have the highest average usable savings,
   approximately LKR 841,984.
   Agent-acquired borrowers have the lowest observed default rate,
   3 of 56 assessed borrowers (5.36%).
   A single snapshot cannot show which channel performs consistently
   over time or which is most profitable.

### Recommendation 1 — Investigate repayment problems before expanding lending

The lending and collections teams should validate the 23 recorded
defaults and review repayment histories, starting with the Starter
segment because of its higher observed default rate.

Use payment records to understand missed payments and identify suitable
follow-up or support. Do not automatically reject applicants based
only on their customer segment.

Run a three-month review and track overdue balances, amounts recovered,
and repayment-status changes among the same starting borrowers.
This prevents new lending from making the default percentage look
better simply by increasing the denominator.

### Recommendation 2 — Test a targeted additional-product campaign

The customer relationship team should review the 251 potential
cross-sell customers for product suitability, verification status,
liquidity needs, and contact preferences.

Offer fixed deposits only where customers can afford to set money
aside. Discuss insurance where it matches their needs.

Start with a small pilot and compare it with a similar group that
does not receive the campaign. Measure additional product uptake,
net financial contribution after campaign costs, and complaints.
Expand only if the pilot produces useful results.

### Additional data needed

1. **Dated transaction and loan repayment records**
   Obtain monthly savings balances, deposits, withdrawals, scheduled
   repayments, actual repayments, and days past due.
   These records would verify savings trends, show whether repayment
   problems are improving, and establish a reliable reporting date.

2. **Customer needs and campaign economics data**
   Obtain customer product preferences, liquidity needs, contact
   permissions, previous offer responses, acquisition costs, and
   product costs and revenue.
   These would help assess suitable offers and compare channels by
   profitability rather than savings balances alone.

## Q6 — Critical Review of My Analysis

### 1. Savings results depend on my treatment of unusual balances
**Severity: High**

I excluded six missing balances, four negative balances, and one
suspicious balance of approximately LKR 1 billion from savings analysis.

However, an unusually large balance is not automatically an error.
Negative balances may also have a valid business explanation.

**Possible impact:**
Among nonnegative recorded balances, including the suspicious balance
increases average savings from approximately LKR 773,720 to
LKR 2,812,958. It may also change group comparisons and business priorities.
The reported savings total is therefore a total of usable balances,
not a verified total for all customers.

**Improvement:**
Check the flagged balances against account records and confirm whether
negative balances are permitted. Retain the original values and compare
results with and without exclusions, including segment-level results.


### 2. The reporting date and measurement periods are unconfirmed
**Severity: High**

I used 1 January 2024 as a diagnostic date when checking ages, but it
is not a confirmed reporting date. Some recorded dates fall after it.

The monthly deposit and withdrawal averages also do not establish
how savings balances changed over time.

**Possible impact:**
Age, account tenure, and recent app activity could be misclassified.
The 100% positive net flow result applies to 492 assessable customers;
it does not prove that every customer's savings increased over time
or that overall financial health is strong.

**Improvement:**
Confirm the snapshot date and the periods used to calculate monthly
averages. Use dated transactions and monthly closing balances to
measure actual trends. Keep the eight unavailable flows visible.


### 3. Some borrower classifications are inferred
**Severity: High**

I used supporting loan information to resolve contradictory loan flags.
The analytical classification contains 221 borrowers, compared with
218 borrowers using the original flags.

**Possible impact:**
Loan penetration changes from 44.04% to 44.65% among known statuses.
Default rates and exposure calculations may also change.
The supporting information could describe an old or closed loan,
rather than a current loan.

**Improvement:**
Validate the classifications against the loan ledger, including
account status and closure dates. Report sensitivity results using
both original flags and analytical classifications.


### 4. Group differences do not establish causes
**Severity: Medium**

My comparisons describe the available customers without controlling
for income, loan age, loan amount, or other differences between groups.

For example, four of 13 Starter Vehicle borrowers are marked Defaulted,
giving a rate of 30.77%. One borrower represents about 7.69 percentage
points in this group, so the percentage is sensitive to a few records.

**Possible impact:**
A group may appear riskier because of its customer mix or loan maturity.
Similarly, differences between app users and non-users do not prove
that using the app changes savings.

**Improvement:**
Show counts beside percentages, quantify uncertainty where appropriate,
and compare similar customers and loan cohorts. Use larger samples
and repeated observations before making broad decisions.


### 5. Outstanding loans divided by savings is a limited risk measure
**Severity: Medium**

The ratio compares outstanding loans with savings held in this dataset.
It does not include income, assets, savings at other institutions,
other debts, or repayment obligations.

**Possible impact:**
The 128 assessable borrowers with ratios above 1.0 should not
automatically be described as unable to repay.
Their LKR 131.77 million in outstanding loans is exposure,
not an estimate of expected loss.

**Improvement:**
Combine this ratio with repayment history, verified income, total
debt obligations, and relevant collateral information.
Use it as a screening indicator rather than a lending decision rule.